# Fine-tuning LaMa + A-ESRGAN para Restauración de Fotografía Vintage

Pipeline de re-entrenamiento supervisado de dos etapas:
1. Fine-tuning de **LaMa** (inpainting) sobre pares degradado/GT del dataset vintage sintético.
2. Fine-tuning de **A-ESRGAN** (super-resolución x4) sobre pares LR/HR generados desde imágenes limpias.
3. Evaluación cuantitativa (PSNR, SSIM, LPIPS) comparando modelos pre-entrenados vs fine-tuneados.

> **Requisito:** Ejecutar en Google Colab con GPU (T4 o superior). Runtime → Change runtime type → T4 GPU.

## 0. Setup: Instalación y Configuración

Instala las dependencias necesarias para todo el pipeline. Reinicia el runtime si Colab ya había cargado versiones anteriores de PIL/numpy antes de ejecutar esta celda.

In [ ]:
# Celda 1: Dependencias base
%pip install -q \
    kagglehub \
    simple-lama-inpainting \
    opencv-python \
    matplotlib \
    numpy \
    tqdm \
    scikit-image \
    pandas \
    lpips \
    omegaconf \
    hydra-core \
    "basicsr>=1.3.3.11" \
    "facexlib>=0.2.0.3" \
    "gfpgan>=0.2.1"

In [ ]:
# Celda 2: Imports y configuración global
%matplotlib inline

import os, sys, site, shutil, subprocess
from pathlib import Path
from typing import Optional, Sequence
import cv2
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import torch
from IPython.display import display

ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from google.colab import output
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WORK_DIR = Path('/content/finetune_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

LAMA_REPO_DIR = Path('/content/lama')
AESRGAN_REPO_DIR = Path('/content/A-ESRGAN')

# Directorios de datos
LAMA_TRAIN_DIR  = WORK_DIR / 'data' / 'lama_train'
LAMA_VAL_DIR    = WORK_DIR / 'data' / 'lama_val'
SR_TRAIN_DIR    = WORK_DIR / 'data' / 'aesrgan_train'
SR_VAL_DIR      = WORK_DIR / 'data' / 'aesrgan_val'

for d in [LAMA_TRAIN_DIR/'images', LAMA_TRAIN_DIR/'masks',
          LAMA_VAL_DIR/'images',   LAMA_VAL_DIR/'masks',
          SR_TRAIN_DIR/'HR',       SR_TRAIN_DIR/'LR',
          SR_VAL_DIR/'HR',         SR_VAL_DIR/'LR',
          WORK_DIR/'checkpoints'/'lama_finetuned',
          WORK_DIR/'checkpoints'/'aesrgan_finetuned',
          WORK_DIR/'results']:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'GPU disponible: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 1. Descarga de Datasets

Se descargan los dos datasets del dominio vintage:

- **`marcinrutecki/old-photos`**: fotos antiguas reales (sin GT pareado; solo para validación visual).
- **`shrutimandaokar2301/vintage-degraded-image-synthetic-real`**: dataset con split limpio/degradado pareado.
  - `01_Clean_Candidates_GT/` — imágenes limpias de referencia (GT)
  - `02_Damaged_Testing_Set/` — imágenes dañadas reales
  - `03_Synthetic_Dataset/Train_Input_Degraded/` — imágenes degradadas sintéticas (pareadas con GT por token `imgNN`)

In [ ]:
# Celda 3: Descarga datasets
import kagglehub

old_photos_path    = Path(kagglehub.dataset_download('marcinrutecki/old-photos'))
vintage_path       = Path(kagglehub.dataset_download(
    'shrutimandaokar2301/vintage-degraded-image-synthetic-real'))

CLEAN_GT_DIR       = vintage_path / '01_Clean_Candidates_GT'
DAMAGED_REAL_DIR   = vintage_path / '02_Damaged_Testing_Set'
SYNTH_DEGRADED_DIR = vintage_path / '03_Synthetic_Dataset' / 'Train_Input_Degraded'

for name, path in [
    ('Old Photos',         old_photos_path),
    ('Clean GT',           CLEAN_GT_DIR),
    ('Damaged Real',       DAMAGED_REAL_DIR),
    ('Synth Degraded',     SYNTH_DEGRADED_DIR),
]:
    files = sorted(p for p in path.rglob('*')
                   if p.suffix.lower() in {'.png','.jpg','.jpeg'})
    print(f'{name:20s}: {len(files):4d} imágenes')

### 1.1 Emparejamiento degradado → GT limpio

Las imágenes sintéticas del split `03_Synthetic_Dataset` incluyen un token `imgNN` en el nombre de fichero que permite emparejarlas con su GT limpio en `01_Clean_Candidates_GT`. La función `build_pairs()` extrae estos pares.

In [ ]:
# Celda 4: Construir mapa de pares degradado -> GT limpio
import re

def extract_img_id(path: Path) -> Optional[str]:
    """Extrae el token imgNN del nombre de fichero."""
    m = re.search(r'img\d+', path.stem.lower())
    return m.group(0) if m else None

def build_pairs(degraded_dir: Path, clean_dir: Path) -> list[dict]:
    """Empareja imágenes degradadas con su GT limpio por token imgNN."""
    clean_files = sorted(p for p in clean_dir.rglob('*')
                         if p.suffix.lower() in {'.png','.jpg','.jpeg'})
    clean_by_id = {extract_img_id(p): p for p in clean_files if extract_img_id(p)}

    pairs = []
    for deg_path in sorted(p for p in degraded_dir.rglob('*')
                            if p.suffix.lower() in {'.png','.jpg','.jpeg'}):
        img_id = extract_img_id(deg_path)
        if img_id and img_id in clean_by_id:
            pairs.append({'degraded': deg_path, 'clean': clean_by_id[img_id]})

    return pairs

SYNTH_PAIRS = build_pairs(SYNTH_DEGRADED_DIR, CLEAN_GT_DIR)
print(f'Pares sintéticos disponibles: {len(SYNTH_PAIRS)}')
assert len(SYNTH_PAIRS) > 0, 'No se encontraron pares. Revisar estructura del dataset.'

In [ ]:
# Celda 5: Visualizar 3 pares de ejemplo
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
for i, pair in enumerate(SYNTH_PAIRS[:3]):
    for j, (key, title) in enumerate([('degraded','Degradada'), ('clean','GT limpio')]):
        img = Image.open(pair[key]).convert('RGB')
        axes[i][j].imshow(img)
        axes[i][j].set_title(f'{title}: {pair[key].name}', fontsize=8)
        axes[i][j].axis('off')
plt.tight_layout()
plt.show()
print('Si los pares se ven coherentes, continúa. De lo contrario revisa build_pairs().')

## 2. Preparación de Datos: LaMa Fine-tuning

LaMa necesita pares (imagen, máscara) donde la máscara es binaria (0=fondo, 255=zona a reconstruir).

Estrategia: comparamos cada imagen degradada sintética con su GT limpio y umbralamos la diferencia absoluta media por canal. Las zonas con diferencia > `DIFF_THRESHOLD` píxeles se marcan como dañadas. Se aplica una dilatación morfológica para cubrir bordes de transición.

Split: 80% train, 20% val, aleatorio con semilla fija.

In [ ]:
# Celda 6: Generación de máscaras automáticas por diferencia imagen
def generate_damage_mask(
    degraded: Image.Image,
    clean: Image.Image,
    diff_threshold: int = 30,
    dilation_px: int = 8,
) -> Image.Image:
    """
    Genera una máscara binaria de las zonas dañadas comparando
    imagen degradada con su GT limpio. Umbral aplicado sobre diferencia en gris.
    Los píxeles blancos (255) indican zona dañada (a reconstruir).
    """
    size = clean.size
    deg_arr   = np.array(degraded.convert('RGB').resize(size)).astype(np.int16)
    clean_arr = np.array(clean.convert('RGB').resize(size)).astype(np.int16)

    diff = np.abs(deg_arr - clean_arr).mean(axis=2).astype(np.uint8)
    _, mask = cv2.threshold(diff, diff_threshold, 255, cv2.THRESH_BINARY)

    # Dilatar para cubrir bordes de transición
    kernel = np.ones((dilation_px, dilation_px), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)

    return Image.fromarray(mask, mode='L')


# Test unitario rápido
_test_deg   = Image.new('RGB', (100, 100), (100, 100, 100))
_test_clean = Image.new('RGB', (100, 100), (200, 200, 200))
_test_mask  = generate_damage_mask(_test_deg, _test_clean, diff_threshold=30)
assert np.array(_test_mask).mean() > 200, 'La máscara debería ser casi toda blanca aquí'
print('generate_damage_mask OK')

In [ ]:
# Celda 7: Crear split 80/20 y copiar datos LaMa
import random
import shutil

random.seed(42)
pairs_shuffled = SYNTH_PAIRS.copy()
random.shuffle(pairs_shuffled)

val_size    = max(1, int(len(pairs_shuffled) * 0.2))
val_pairs   = pairs_shuffled[:val_size]
train_pairs = pairs_shuffled[val_size:]

print(f'Train: {len(train_pairs)} pares | Val: {len(val_pairs)} pares')

DIFF_THRESHOLD  = 30   # Umbral de diferencia para generar máscara
DILATION_PX     = 8    # Dilatación de máscara en píxeles

def prepare_lama_split(pairs: list[dict], images_dir: Path, masks_dir: Path, desc: str):
    skipped = 0
    for pair in pairs:
        try:
            deg   = Image.open(pair['degraded']).convert('RGB')
            clean = Image.open(pair['clean']).convert('RGB')
            mask  = generate_damage_mask(deg, clean, DIFF_THRESHOLD, DILATION_PX)

            # Solo incluir si la máscara tiene al menos 1% de píxeles dañados
            mask_arr = np.array(mask)
            if mask_arr.mean() < 2.55:   # < 1%
                skipped += 1
                continue

            stem = pair['degraded'].stem
            deg.save(images_dir / f'{stem}.png')
            mask.save(masks_dir  / f'{stem}_mask.png')
        except Exception as e:
            print(f'  [WARN] {pair["degraded"].name}: {e}')
            skipped += 1

    total = len(pairs) - skipped
    print(f'{desc}: {total} pares válidos guardados ({skipped} omitidos)')

prepare_lama_split(train_pairs, LAMA_TRAIN_DIR/'images', LAMA_TRAIN_DIR/'masks', 'LaMa Train')
prepare_lama_split(val_pairs,   LAMA_VAL_DIR/'images',   LAMA_VAL_DIR/'masks',   'LaMa Val')

In [ ]:
# Celda 8: Visualizar 3 máscaras generadas
sample_images = sorted((LAMA_TRAIN_DIR/'images').glob('*.png'))[:3]
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, img_path in enumerate(sample_images):
    mask_path = LAMA_TRAIN_DIR / 'masks' / img_path.name.replace('.png', '_mask.png')
    img  = Image.open(img_path).convert('RGB')
    mask = Image.open(mask_path).convert('L')
    overlay = np.array(img.copy())
    overlay[np.array(mask) > 127] = [255, 0, 0]
    axes[i][0].imshow(img);                axes[i][0].set_title('Degradada');  axes[i][0].axis('off')
    axes[i][1].imshow(mask, cmap='gray');  axes[i][1].set_title('Máscara');    axes[i][1].axis('off')
    axes[i][2].imshow(overlay);            axes[i][2].set_title('Overlay');    axes[i][2].axis('off')
plt.tight_layout()
plt.show()
print('Verifica: la máscara (rojo) cubre las zonas dañadas visibles.')

## 3. Setup LaMa: Repositorio y Configuración

Se clona el repositorio oficial de LaMa (`advimman/lama`), se instalan sus dependencias y se descargan los pesos pre-entrenados `big-lama.pt` desde HuggingFace. También se genera el fichero YAML de configuración para el fine-tuning.

In [ ]:
# Celda 9: Clonar LaMa oficial (advimman/lama)
LAMA_REPO_URL = 'https://github.com/advimman/lama.git'

if not LAMA_REPO_DIR.exists():
    subprocess.run(['git', 'clone', LAMA_REPO_URL, str(LAMA_REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(LAMA_REPO_DIR), 'pull'], check=True)

# Instalar dependencias LaMa
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(LAMA_REPO_DIR / 'requirements.txt')
], check=True)

print(f'LaMa repo: {LAMA_REPO_DIR}')
print('Contenido:', list(LAMA_REPO_DIR.iterdir())[:8])

In [ ]:
# Celda 10: Descargar pesos LaMa pre-entrenados (big-lama)
LAMA_WEIGHTS_DIR = LAMA_REPO_DIR / 'big-lama'
LAMA_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
LAMA_MODEL_PATH  = LAMA_WEIGHTS_DIR / 'models' / 'best.ckpt'

(LAMA_WEIGHTS_DIR / 'models').mkdir(parents=True, exist_ok=True)

# Pesos big-lama desde HuggingFace
if not LAMA_MODEL_PATH.exists():
    subprocess.run([
        'wget', '-q', '-O', str(LAMA_MODEL_PATH),
        'https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.pt'
    ], check=True)
    print('Pesos big-lama descargados')
else:
    print('Pesos ya disponibles:', LAMA_MODEL_PATH)

print('Tamaño:', LAMA_MODEL_PATH.stat().st_size // (1024*1024), 'MB')

In [ ]:
# Celda 11: Crear config YAML para fine-tuning LaMa
LAMA_FINETUNE_CONFIG = WORK_DIR / 'lama_finetune_config.yaml'

lama_config_content = f"""
trainer:
  kwargs:
    max_epochs: 10
    gpus: 1
    precision: 16
    log_every_n_steps: 5

optimizers:
  generator:
    kind: adam
    lr: 0.0001

losses:
  l1:
    weight: 1.0
  perceptual:
    weight: 0.1
    networks: [vgg16]

data:
  batch_size: 4
  val_batch_size: 2
  num_workers: 2
  train:
    indir: {LAMA_TRAIN_DIR}
    out_size: 256
    mask_settings:
      mask_mode: from_file
      masks_dir: {LAMA_TRAIN_DIR / 'masks'}
  val:
    indir: {LAMA_VAL_DIR}
    out_size: 256
    mask_settings:
      mask_mode: from_file
      masks_dir: {LAMA_VAL_DIR / 'masks'}

generator:
  kind: ffc_resnet
  input_nc: 4
  output_nc: 3
  ngf: 64
  n_downsampling: 3
  n_blocks: 18
  add_out_act: sigmoid
  init_conv_kwargs:
    ratio_gin: 0
    ratio_gout: 0
  downsample_conv_kwargs:
    ratio_gin: 0
    ratio_gout: 0
  resnet_conv_kwargs:
    ratio_gin: 0.75
    ratio_gout: 0.75

discriminator:
  kind: patchgan

training:
  visualize_each_iters: 200
  val_visualize_at_start: true

checkpoint:
  monitor: val/l1
  filename: 'epoch{{epoch:02d}}-loss{{val/l1:.4f}}'
  save_top_k: 3
  mode: min
"""

LAMA_FINETUNE_CONFIG.write_text(lama_config_content.strip())
print(f'Config guardada en: {LAMA_FINETUNE_CONFIG}')

## 4. Fine-tuning LaMa

### Opción 1 (principal): CLI de LaMa con Hydra
Se invoca `bin/train.py` pasando los overrides de Hydra por línea de comandos.

### Opción 2 (fallback): Bucle manual PyTorch
Si la API Hydra del repositorio es incompatible con la versión instalada en Colab, usar el bucle PyTorch de la celda FALLBACK. Este enfoque carga el modelo de `simple_lama_inpainting` y lo fine-tunea con L1 loss sobre la zona enmascarada.

> Ejecuta primero la celda de parches y luego la celda principal. Si falla con error de Hydra/config, ejecuta en su lugar la celda FALLBACK.

In [ ]:
# Celda 12: Añadir LaMa al sys.path y aplicar parches de compatibilidad
if str(LAMA_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(LAMA_REPO_DIR))

# Parche basicsr: torchvision >= 0.16 eliminó functional_tensor
for base in site.getsitepackages() + [site.getusersitepackages()]:
    p = Path(base) / 'basicsr' / 'data' / 'degradations.py'
    if p.exists():
        text = p.read_text()
        old  = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        new  = 'from torchvision.transforms.functional import rgb_to_grayscale'
        if old in text:
            p.write_text(text.replace(old, new))
            print(f'Parche aplicado: {p}')

# Parche PIL._typing para lpips/torchvision
import PIL._typing
if not hasattr(PIL._typing, '_Ink'):
    from typing import Union
    PIL._typing._Ink = Union[int, tuple]

print('Entorno LaMa listo')

In [ ]:
# Celda 13: Lanzar fine-tuning LaMa (opción principal via CLI Hydra)
MAX_EPOCHS    = 10
LAMA_CKPT_DIR = WORK_DIR / 'checkpoints' / 'lama_finetuned'

lama_train_cmd = [
    sys.executable,
    str(LAMA_REPO_DIR / 'bin' / 'train.py'),
    f'trainer.kwargs.max_epochs={MAX_EPOCHS}',
    f'data.train.indir={LAMA_TRAIN_DIR}',
    f'data.val.indir={LAMA_VAL_DIR}',
    f'data.train.mask_settings.masks_dir={LAMA_TRAIN_DIR / "masks"}',
    f'data.val.mask_settings.masks_dir={LAMA_VAL_DIR / "masks"}',
    f'location={LAMA_CKPT_DIR}',
    '--config-path', str(LAMA_REPO_DIR / 'configs' / 'training'),
    '--config-name', 'lama-fourier',
]

print('Iniciando fine-tuning LaMa...')
print(' '.join(str(x) for x in lama_train_cmd))

result = subprocess.run(
    lama_train_cmd,
    cwd=str(LAMA_REPO_DIR),
    capture_output=False,
)

if result.returncode != 0:
    print('[ERROR] El entrenamiento falló. Revisar logs arriba. Usa la celda FALLBACK.')
else:
    print('[OK] Fine-tuning LaMa completado.')
    checkpoints = sorted(LAMA_CKPT_DIR.rglob('*.ckpt'))
    print(f'Checkpoints generados: {[c.name for c in checkpoints]}')

In [ ]:
# Celda 14 (FALLBACK): Bucle de fine-tuning manual LaMa
# Usar SOLO si la celda anterior falla con error de Hydra/config.

from simple_lama_inpainting import SimpleLama
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

class LamaFineTuneDataset(Dataset):
    def __init__(self, images_dir: Path, masks_dir: Path, img_size: int = 256):
        self.img_paths = sorted(images_dir.glob('*.png'))
        self.masks_dir = masks_dir
        self.img_size  = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path  = self.img_paths[idx]
        mask_path = self.masks_dir / img_path.name.replace('.png', '_mask.png')
        img  = self.transform(Image.open(img_path).convert('RGB'))
        mask = transforms.Resize(
            (self.img_size, self.img_size),
            interpolation=transforms.InterpolationMode.NEAREST
        )(transforms.ToTensor()(Image.open(mask_path).convert('L')))
        masked_img = img * (1 - mask)
        return masked_img, mask, img

train_ds = LamaFineTuneDataset(LAMA_TRAIN_DIR / 'images', LAMA_TRAIN_DIR / 'masks')
val_ds   = LamaFineTuneDataset(LAMA_VAL_DIR   / 'images', LAMA_VAL_DIR   / 'masks')
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

simple_lama = SimpleLama()
model = simple_lama.model.to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.L1Loss()

FALLBACK_EPOCHS = 10
best_val_loss   = float('inf')
LAMA_BEST_CKPT  = WORK_DIR / 'checkpoints' / 'lama_finetuned' / 'lama_best.pth'

for epoch in range(FALLBACK_EPOCHS):
    model.train()
    train_loss = 0.0
    for masked_img, mask, gt in train_dl:
        masked_img, mask, gt = masked_img.to(DEVICE), mask.to(DEVICE), gt.to(DEVICE)
        inp  = torch.cat([masked_img, mask], dim=1)
        pred = model(inp)
        loss = criterion(pred * mask, gt * mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for masked_img, mask, gt in val_dl:
            masked_img, mask, gt = masked_img.to(DEVICE), mask.to(DEVICE), gt.to(DEVICE)
            inp  = torch.cat([masked_img, mask], dim=1)
            pred = model(inp)
            val_loss += criterion(pred * mask, gt * mask).item()

    train_loss /= len(train_dl)
    val_loss   /= len(val_dl)
    print(f'Epoch {epoch+1:02d}/{FALLBACK_EPOCHS} | train_l1={train_loss:.4f} | val_l1={val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), LAMA_BEST_CKPT)
        print(f'  -> Checkpoint guardado: val_l1={val_loss:.4f}')

print(f'Fine-tuning LaMa completado. Mejor val_l1={best_val_loss:.4f}')
print(f'Checkpoint: {LAMA_BEST_CKPT}')

## 5. Preparación de Datos: A-ESRGAN Fine-tuning

A-ESRGAN aprende super-resolución x4: necesita pares HR (imagen limpia) / LR (imagen downscaled x4).

Estrategia: HR = GT limpio del dataset vintage. LR = HR redimensionado con Lanczos x4. Se garantiza que el tamaño HR sea divisible por 4 (recortando píxeles sobrantes).

Las imágenes HR con lado menor a `MIN_HR_SIZE = 128` px se descartan.

In [ ]:
# Celda 15: Crear pares LR/HR para A-ESRGAN
SCALE_FACTOR = 4
MIN_HR_SIZE  = 128   # Descartar imágenes HR con lado menor a MIN_HR_SIZE px

def prepare_sr_split(pairs: list[dict], hr_dir: Path, lr_dir: Path, desc: str):
    skipped = 0
    for pair in pairs:
        try:
            hr = Image.open(pair['clean']).convert('RGB')

            w, h = hr.size
            if w < MIN_HR_SIZE or h < MIN_HR_SIZE:
                skipped += 1
                continue

            # Asegurar que HR es divisible por SCALE_FACTOR
            new_w = (w // SCALE_FACTOR) * SCALE_FACTOR
            new_h = (h // SCALE_FACTOR) * SCALE_FACTOR
            hr = hr.crop((0, 0, new_w, new_h))

            # LR = HR downscaled x4 con Lanczos
            lr_from_hr = hr.resize(
                (new_w // SCALE_FACTOR, new_h // SCALE_FACTOR),
                Image.Resampling.LANCZOS
            )

            stem = pair['clean'].stem
            hr.save(hr_dir / f'{stem}.png')
            lr_from_hr.save(lr_dir / f'{stem}.png')
        except Exception as e:
            print(f'  [WARN] {pair["clean"].name}: {e}')
            skipped += 1

    total = len(pairs) - skipped
    print(f'{desc}: {total} pares HR/LR guardados ({skipped} omitidos)')

prepare_sr_split(train_pairs, SR_TRAIN_DIR / 'HR', SR_TRAIN_DIR / 'LR', 'SR Train')
prepare_sr_split(val_pairs,   SR_VAL_DIR   / 'HR', SR_VAL_DIR   / 'LR', 'SR Val')

In [ ]:
# Celda 16: Verificar splits
for split, hr_dir, lr_dir in [
    ('Train', SR_TRAIN_DIR / 'HR', SR_TRAIN_DIR / 'LR'),
    ('Val',   SR_VAL_DIR   / 'HR', SR_VAL_DIR   / 'LR'),
]:
    hr_files = sorted(hr_dir.glob('*.png'))
    lr_files = sorted(lr_dir.glob('*.png'))
    assert len(hr_files) == len(lr_files), 
        f'Mismatch {split}: HR={len(hr_files)} LR={len(lr_files)}'
    for f in random.sample(hr_files, min(2, len(hr_files))):
        hr = Image.open(f)
        lr = Image.open(lr_dir / f.name)
        expected_lr = (hr.width // SCALE_FACTOR, hr.height // SCALE_FACTOR)
        assert lr.size == expected_lr, 
            f'Escala incorrecta: HR={hr.size} LR={lr.size}'
    print(f'{split}: {len(hr_files)} pares HR/LR verificados OK')

## 6. Fine-tuning A-ESRGAN

Se clona el repositorio A-ESRGAN y se generan pesos fine-tuneados partiendo de `A_ESRGAN_Single.pth` como inicialización (`pretrain_network_g`, `strict_load_g: false`).

El entrenamiento usa la configuración BasicSR (formato `.yml`): modelo `RealESRGANModel`, red `RRDBNet` (23 bloques RRDB, escala x4), discriminador `UNetDiscriminatorSN`, losses L1 + perceptual VGG19 + GAN. Se limita a 2000 iteraciones para esta primera versión.

In [ ]:
# Celda 17: Clonar A-ESRGAN y descargar pesos pre-entrenados
AESRGAN_REPO_URL   = 'https://github.com/stroking-fishes-ml-corp/A-ESRGAN.git'
AESRGAN_MODEL_URL  = 'https://github.com/stroking-fishes-ml-corp/A-ESRGAN/releases/download/v1.0.0/A_ESRGAN_Single.pth'
AESRGAN_PRETRAINED = AESRGAN_REPO_DIR / 'experiments' / 'pretrained_models' / 'A_ESRGAN_Single.pth'

if not AESRGAN_REPO_DIR.exists():
    subprocess.run(['git', 'clone', AESRGAN_REPO_URL, str(AESRGAN_REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(AESRGAN_REPO_DIR), 'pull'], check=True)

AESRGAN_PRETRAINED.parent.mkdir(parents=True, exist_ok=True)
if not AESRGAN_PRETRAINED.exists():
    subprocess.run(['wget', '-q', '-O', str(AESRGAN_PRETRAINED), AESRGAN_MODEL_URL], check=True)
    print('Pesos A-ESRGAN descargados')
else:
    print('Pesos ya disponibles:', AESRGAN_PRETRAINED)

# Listar scripts de entrenamiento disponibles
train_scripts = sorted(AESRGAN_REPO_DIR.rglob('train*.py'))
print('Scripts de entrenamiento encontrados:')
for s in train_scripts:
    print(f'  {s.relative_to(AESRGAN_REPO_DIR)}')

In [ ]:
# Celda 18: Crear YAML de entrenamiento A-ESRGAN (formato BasicSR)
AESRGAN_FINETUNE_CONFIG = WORK_DIR / 'aesrgan_finetune.yml'
AESRGAN_CKPT_DIR        = WORK_DIR / 'checkpoints' / 'aesrgan_finetuned'

aesrgan_config_content = f"""
# General settings
name: AESRGAN_finetune_vintage
model_type: RealESRGANModel
scale: 4
num_gpu: 1
manual_seed: 42

# Datasets
datasets:
  train:
    name: vintage_train
    type: RealESRGANDataset
    dataroot_gt: {SR_TRAIN_DIR / 'HR'}
    dataroot_lq: {SR_TRAIN_DIR / 'LR'}
    io_backend:
      type: disk
    gt_size: 256
    use_hflip: true
    use_rot: true
    use_shuffle: true
    num_worker_per_gpu: 2
    batch_size_per_gpu: 4
    dataset_enlarge_ratio: 1
    prefetch_mode: ~
  val:
    name: vintage_val
    type: PairedImageDataset
    dataroot_gt: {SR_VAL_DIR / 'HR'}
    dataroot_lq: {SR_VAL_DIR / 'LR'}
    io_backend:
      type: disk

# Network structures
network_g:
  type: RRDBNet
  num_in_ch: 3
  num_out_ch: 3
  num_feat: 64
  num_block: 23
  num_grow_ch: 32
  scale: 4

network_d:
  type: UNetDiscriminatorSN
  num_in_ch: 3
  num_feat: 64
  skip_connection: true

# Path
path:
  pretrain_network_g: {AESRGAN_PRETRAINED}
  strict_load_g: false
  resume_state: ~

# Training settings
train:
  ema_decay: 0.999
  optim_g:
    type: Adam
    lr: !!float 1e-4
    weight_decay: 0
    betas: [0.9, 0.99]
  optim_d:
    type: Adam
    lr: !!float 1e-4
    weight_decay: 0
    betas: [0.9, 0.99]
  scheduler:
    type: MultiStepLR
    milestones: [400000]
    gamma: 0.5
  total_iter: 2000
  warmup_iter: -1
  pixel_opt:
    type: L1Loss
    loss_weight: 1.0
    reduction: mean
  perceptual_opt:
    type: PerceptualLoss
    layer_weights:
      'conv1_2': 0.1
      'conv2_2': 0.1
      'conv3_4': 1
      'conv4_4': 1
      'conv5_4': 1
    vgg_type: vgg19
    use_input_norm: true
    perceptual_weight: !!float 1.0
    style_weight: 0
    range_norm: false
    criterion: l1
  gan_opt:
    type: GANLoss
    gan_type: vanilla
    real_label_val: 1.0
    fake_label_val: 0.0
    loss_weight: !!float 1e-1
  net_d_iters: 1
  net_d_init_iters: 0

# Validation
val:
  val_freq: !!float 200
  save_img: true
  metrics:
    psnr:
      type: calculate_psnr
      crop_border: 4
      test_y_channel: false

# Logging
logger:
  print_freq: 50
  save_checkpoint_freq: !!float 500
  use_tb_logger: false

# dist training settings
dist_params:
  backend: nccl
  port: 29500
"""

AESRGAN_FINETUNE_CONFIG.write_text(aesrgan_config_content.strip())
print(f'Config A-ESRGAN guardada: {AESRGAN_FINETUNE_CONFIG}')

In [ ]:
# Celda 19: Lanzar fine-tuning A-ESRGAN
if str(AESRGAN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(AESRGAN_REPO_DIR))

# Buscar script de entrenamiento en orden de preferencia
train_script = AESRGAN_REPO_DIR / 'basicsr' / 'train.py'
if not train_script.exists():
    train_script = AESRGAN_REPO_DIR / 'realesrgan' / 'train.py'
if not train_script.exists():
    train_script = Path(site.getsitepackages()[0]) / 'basicsr' / 'train.py'

print(f'Usando script: {train_script}')

aesrgan_train_cmd = [
    sys.executable, str(train_script),
    '-opt', str(AESRGAN_FINETUNE_CONFIG),
]

print('Iniciando fine-tuning A-ESRGAN...')
result = subprocess.run(
    aesrgan_train_cmd,
    cwd=str(AESRGAN_REPO_DIR),
    capture_output=False,
)

if result.returncode != 0:
    print('[ERROR] Fine-tuning A-ESRGAN falló. Ver logs arriba.')
else:
    print('[OK] Fine-tuning A-ESRGAN completado.')
    ckpts = sorted(AESRGAN_REPO_DIR.rglob('net_g_*.pth'))
    print(f'Checkpoints: {[c.name for c in ckpts]}')
    if ckpts:
        best_ckpt = ckpts[-1]
        shutil.copy(best_ckpt, AESRGAN_CKPT_DIR / best_ckpt.name)
        print(f'Último checkpoint copiado a: {AESRGAN_CKPT_DIR / best_ckpt.name}')

## 7. Evaluación: Pre-entrenado vs Fine-tuneado

Se evalúan hasta `N_EVAL = 5` pares del conjunto de validación con métricas full-reference:
- **PSNR** (↑ mejor): relación señal-ruido en dB.
- **SSIM** (↑ mejor): similitud estructural.
- **LPIPS** (↓ mejor): similitud perceptual con AlexNet.

Se compara el pipeline completo **Pre-entrenado** (LaMa + A-ESRGAN con pesos originales) frente al **Fine-tuneado** (si los checkpoints existen). Los resultados se guardan en CSV y se muestra una comparativa visual.

In [ ]:
# Celda 20: Funciones de inferencia para ambas versiones
from simple_lama_inpainting import SimpleLama

def run_lama_inference(
    image: Image.Image,
    mask: Image.Image,
    model_state_dict_path: Optional[Path] = None,
) -> Image.Image:
    """
    Ejecuta LaMa inpainting. Si model_state_dict_path es None, usa pesos
    pre-entrenados por defecto. Si se pasa una ruta .pth, carga ese state_dict.
    """
    lama = SimpleLama()
    if model_state_dict_path is not None:
        state = torch.load(model_state_dict_path, map_location=DEVICE)
        lama.model.load_state_dict(state, strict=False)
        lama.model.to(DEVICE).eval()
    return lama(image, mask).convert('RGB')


def run_aesrgan_inference(
    image: Image.Image,
    model_path: Path,
    tile: int = 400,
) -> Image.Image:
    """
    Ejecuta A-ESRGAN guardando temporalmente la imagen de entrada y
    llamando al script de inferencia con el model_path indicado.
    """
    tmp_input  = WORK_DIR / 'results' / '_tmp_aesrgan_input.png'
    tmp_output = WORK_DIR / 'results' / '_tmp_aesrgan_out'
    tmp_output.mkdir(exist_ok=True)
    image.save(tmp_input)

    cmd = [
        sys.executable,
        str(AESRGAN_REPO_DIR / 'inference_aesrgan.py'),
        '--model_path', str(model_path),
        '--input',      str(tmp_input),
        '--output',     str(tmp_output),
        '--suffix',     'eval',
        '--tile',       str(tile),
    ]
    if DEVICE == 'cuda':
        cmd.append('--half')

    subprocess.run(cmd, cwd=str(AESRGAN_REPO_DIR), check=True)

    out_files = sorted(tmp_output.glob('*.png'))
    assert out_files, f'A-ESRGAN no generó salida en {tmp_output}'
    result = Image.open(out_files[-1]).convert('RGB')
    return result

In [ ]:
# Celda 21: Evaluación comparativa sobre pares de validación
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import lpips as lpips_lib

_LPIPS_FN = lpips_lib.LPIPS(net='alex').eval().to(DEVICE)

def pil_to_lpips_t(img: Image.Image, size=None) -> torch.Tensor:
    from torchvision.transforms.functional import to_tensor
    img = img.convert('RGB')
    if size:
        img = img.resize(size, Image.Resampling.LANCZOS)
    return (to_tensor(img) * 2.0 - 1.0).unsqueeze(0).to(DEVICE)

def compute_all_metrics(pred: Image.Image, ref: Image.Image, name: str) -> dict:
    size = pred.size
    p = np.array(pred.convert('RGB').resize(size)).astype(np.float32)
    r = np.array(ref.convert('RGB').resize(size)).astype(np.float32)
    psnr = peak_signal_noise_ratio(r, p, data_range=255)
    ssim = structural_similarity(r, p, channel_axis=2, data_range=255)
    with torch.no_grad():
        lpips_val = float(_LPIPS_FN(pil_to_lpips_t(pred), pil_to_lpips_t(ref, size)).item())
    return {'model': name, 'PSNR': round(psnr, 4), 'SSIM': round(ssim, 4), 'LPIPS': round(lpips_val, 4)}

# Identificar checkpoints fine-tuneados
lama_ft_ckpts     = sorted((WORK_DIR / 'checkpoints' / 'lama_finetuned').glob('*.pth'))
LAMA_FINETUNE_CKPT = lama_ft_ckpts[-1] if lama_ft_ckpts else None
if LAMA_FINETUNE_CKPT is None:
    print('[WARN] No se encontró checkpoint LaMa fine-tuneado. Solo se evalúa el pre-entrenado.')

aesrgan_ft_ckpts      = sorted((WORK_DIR / 'checkpoints' / 'aesrgan_finetuned').glob('*.pth'))
AESRGAN_FINETUNE_CKPT = aesrgan_ft_ckpts[-1] if aesrgan_ft_ckpts else None

results_rows = []
N_EVAL = min(5, len(val_pairs))

for pair in val_pairs[:N_EVAL]:
    deg   = Image.open(pair['degraded']).convert('RGB')
    clean = Image.open(pair['clean']).convert('RGB')
    mask  = generate_damage_mask(deg, clean, DIFF_THRESHOLD, DILATION_PX)

    # Pipeline pre-entrenado
    lama_pre = run_lama_inference(deg, mask)
    sr_pre   = run_aesrgan_inference(lama_pre, AESRGAN_PRETRAINED)
    results_rows.append(compute_all_metrics(sr_pre, clean, 'Pre-entrenado'))

    # Pipeline fine-tuneado (si disponible)
    if LAMA_FINETUNE_CKPT and AESRGAN_FINETUNE_CKPT:
        lama_ft = run_lama_inference(deg, mask, LAMA_FINETUNE_CKPT)
        sr_ft   = run_aesrgan_inference(lama_ft, AESRGAN_FINETUNE_CKPT)
        results_rows.append(compute_all_metrics(sr_ft, clean, 'Fine-tuneado'))

import pandas as pd
df_results = pd.DataFrame(results_rows)
df_summary = df_results.groupby('model')[['PSNR', 'SSIM', 'LPIPS']].mean().round(4)
print(f'=== Resultados Medios (N={N_EVAL} imágenes) ===')
display(df_summary)

CSV_OUT = WORK_DIR / 'results' / 'eval_pretrained_vs_finetuned.csv'
df_results.to_csv(CSV_OUT, index=False)
print(f'Tabla guardada: {CSV_OUT}')

In [ ]:
# Celda 22: Visualización comparativa
pair_demo = val_pairs[0]
deg   = Image.open(pair_demo['degraded']).convert('RGB')
clean = Image.open(pair_demo['clean']).convert('RGB')
mask  = generate_damage_mask(deg, clean, DIFF_THRESHOLD, DILATION_PX)

lama_pre = run_lama_inference(deg, mask)
sr_pre   = run_aesrgan_inference(lama_pre, AESRGAN_PRETRAINED)

items = [
    ('GT Limpio',    clean),
    ('Degradada',    deg),
    ('Máscara',      mask.convert('RGB')),
    ('LaMa pre',     lama_pre),
    ('Pipeline pre', sr_pre),
]

if LAMA_FINETUNE_CKPT and AESRGAN_FINETUNE_CKPT:
    lama_ft = run_lama_inference(deg, mask, LAMA_FINETUNE_CKPT)
    sr_ft   = run_aesrgan_inference(lama_ft, AESRGAN_FINETUNE_CKPT)
    items += [('LaMa ft', lama_ft), ('Pipeline ft', sr_ft)]

fig, axes = plt.subplots(1, len(items), figsize=(5 * len(items), 6))
for ax, (title, img) in zip(axes, items):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig(WORK_DIR / 'results' / 'comparison_pre_vs_ft.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparativa guardada.')

## 8. Conclusiones y Próximos Pasos

In [ ]:
# Celda 23: Resumen de artefactos generados
print('=== Artefactos generados ===')
for path in sorted(WORK_DIR.rglob('*')):
    if path.is_file():
        size_kb = path.stat().st_size // 1024
        print(f'  {path.relative_to(WORK_DIR)} ({size_kb} KB)')

print("""
=== Limitaciones de esta v1 ===
1. LaMa se fine-tunea con L1 loss simplificado (sin GAN ni perceptual loss completo).
2. Las máscaras son automáticas por diferencia; en imágenes reales se necesitan máscaras manuales o un detector de daños.
3. El split 80/20 usa un dataset pequeño; ampliar con old-photos (sin GT) requiere self-supervised o pseudo-GT.
4. A-ESRGAN usa pares LR/HR generados por downscaling bicúbico, no por degradación sintética de dominio vintage.

=== Próximos pasos sugeridos ===
1. Aumentar epochs a 50-100 y comparar curvas de validación.
2. Implementar generación de LR sintético con degradaciones de dominio vintage (ruido de película, halos, desenfoque).
3. Añadir detector automático de daños (e.g., segmentación con SAM) para máscaras más precisas.
4. Probar con todo el split 02_Damaged_Testing_Set usando pseudo-GT generado por el modelo pre-entrenado.
5. Evaluar FID (Fréchet Inception Distance) para medir calidad perceptual global.
""")